# 04 · Reshape and transpose real images / Reshape y transposición de imágenes reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669">PART III · EXERCISE · 15 MIN</span>

This notebook teaches one idea that prevents many silent bugs:

> **Changing a tensor's shape is not the same as moving its axes.**

We will use real images to see when `transpose` is the correct operation and why `reshape` can produce the expected numbers in `.shape` while giving the wrong interpretation.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">Este cuaderno enseña una idea que evita muchos errores silenciosos:</div><div style="margin:0 0 .7em"><b>Cambiar la forma de un tensor no es lo mismo que mover sus ejes.</b></div><div style="margin:0 0 0">Usaremos imágenes reales para entender cuándo <code>transpose</code> es la operación correcta y por qué <code>reshape</code> puede producir la <code>.shape</code> esperada y, aun así, dar una interpretación equivocada.</div></div>

## What you will be able to do / Lo que podrás hacer

- Read `HWC`, `CHW`, `NHWC`, and `NCHW` as simple sentences.
- Convert a real RGB image from `(H, W, C)` to `(C, H, W)` with `np.transpose`.
- Build a real batch from three different RGB images and convert `NHWC → NCHW`.
- Explain why two axes can have the same size but different meanings.
- Demonstrate visually why `reshape` cannot replace `transpose` when axis meaning must move.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Leer <code>HWC</code>, <code>CHW</code>, <code>NHWC</code> y <code>NCHW</code> como frases sencillas.</li><li style="margin:.35em 0">Convertir una imagen RGB real de <code>(H, W, C)</code> a <code>(C, H, W)</code> con <code>np.transpose</code>.</li><li style="margin:.35em 0">Construir un lote real con tres imágenes RGB distintas y convertir <code>NHWC → NCHW</code>.</li><li style="margin:.35em 0">Explicar por qué dos ejes pueden tener el mismo tamaño y significados diferentes.</li><li style="margin:.35em 0">Demostrar visualmente por qué <code>reshape</code> no puede sustituir <code>transpose</code> cuando debe cambiar la posición de los ejes.</li></ul></div>

## Four axis letters / Cuatro letras de ejes

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

| Letter / Letra | English | Español |
|---|---|---|
| `N` | number of examples / batch | número de ejemplos / lote |
| `H` | height | alto |
| `W` | width | ancho |
| `C` | colour channels | canales de color |

Read every convention as a sentence.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE FOUR CONVENTIONS · LAS CUATRO CONVENCIONES</div><div style="margin:.55em 0"><code>HWC</code> — height × width × colour</div><div style="margin:.55em 0"><code>CHW</code> — colour × height × width</div><div style="margin:.55em 0"><code>NHWC</code> — examples × height × width × colour</div><div style="margin:.55em 0"><code>NCHW</code> — examples × colour × height × width</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><code>HWC</code> — alto × ancho × color</li><li style="margin:.35em 0"><code>CHW</code> — color × alto × ancho</li><li style="margin:.35em 0"><code>NHWC</code> — ejemplos × alto × ancho × color</li><li style="margin:.35em 0"><code>NCHW</code> — ejemplos × color × alto × ancho</li></ul></div>

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Real images shipped with `scikit-image`: histology, microscopy, the astronaut,
and a cup of coffee.

No synthetic pixels anywhere in the exercises.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Imágenes reales incluidas en <code>scikit-image</code>: histología, microscopía, la astronauta y una taza de café. Ningún píxel sintético en los ejercicios.</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from skimage import data

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real images distributed with scikit-image.
photo = data.immunohistochemistry()   # (512, 512, 3), RGB histology
cells_img = data.cell()               # (660, 550), grayscale microscopy
astronaut = data.astronaut()          # (512, 512, 3), RGB photograph
coffee = data.coffee()                # (400, 600, 3), RGB photograph

def center_crop_rgb(img, size=256):
    # Deterministic centre crop so different real RGB images can be stacked.
    h, w, c = img.shape

    if c != 3 or h < size or w < size:
        raise ValueError(
            f"expected RGB image at least {size}x{size}, got {img.shape}"
        )

    r0 = (h - size) // 2
    c0 = (w - size) // 2

    return img[r0:r0 + size, c0:c0 + size]

rgb_sources = [photo, astronaut, coffee]
rgb_names = [
    "Histology / Histología",
    "Astronaut / Astronauta",
    "Coffee / Café",
]

print("Histology / Histología:", photo.shape)
print("Microscopy / Microscopía:", cells_img.shape)
print("Astronaut / Astronauta:", astronaut.shape)
print("Coffee / Café:", coffee.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## NHWC or NCHW / NHWC o NCHW

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

One framework expects `NHWC`. Another expects `NCHW`. A tensor can hold every
correct number and still be read wrongly, because the axes sit in the wrong
places.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">THE DANGEROUS PART · LO PELIGROSO</div>There is <b>no error message</b>. The code runs. The meaning does not.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un framework espera <code>NHWC</code> y otro <code>NCHW</code>. Un tensor puede contener todos los números correctos y aun así interpretarse mal.<br><br>Lo peligroso: <b>no hay mensaje de error</b>. El código corre; el significado no.</div>

### Convention translator / Traductor de convenciones

Pick a convention. Read what every position means.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una convención y observa qué significa cada posición.</div>

In [ ]:
#@title 🔤 Convention translator / Traductor de convenciones — run me / ejecútame { display-mode: 'form' }

convention = widgets.ToggleButtons(
    options=["HWC", "CHW", "NHWC", "NCHW"],
    value="HWC",
    description="Convention / Convención:",
    style={"description_width": "145px"},
)

def explain_convention(value):
    meanings = {
        "HWC": (
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "CHW": (
            "(C, H, W)",
            "colour × height × width",
            "color × alto × ancho",
        ),
        "NHWC": (
            "(N, H, W, C)",
            "examples × height × width × colour",
            "ejemplos × alto × ancho × color",
        ),
        "NCHW": (
            "(N, C, H, W)",
            "examples × colour × height × width",
            "ejemplos × color × alto × ancho",
        ),
    }

    symbols, en, es = meanings[value]

    print("Symbols / Símbolos:", symbols)
    print("EN:", en)
    print("ES:", es)

convention_output = widgets.interactive_output(
    explain_convention,
    {"value": convention},
)

display(widgets.VBox([convention, convention_output]))

## Exercise 1 — one real image, two orderings / Ejercicio 1 — una imagen real, dos órdenes

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

`photo` is a real RGB histology image, <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(H, W, C) = (512, 512, 3)</span>.

`cells_img` is a real grayscale microscopy image, <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(H, W) = (660, 550)</span>.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Which image has a colour axis?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>What should <code>photo.shape</code> become after <code>HWC → CHW</code>?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Which old axis becomes the new axis 0?</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> ¿cuál imagen tiene eje de color? <b>2 ·</b> ¿qué forma tendrá <code>photo</code> tras <code>HWC → CHW</code>? <b>3 ·</b> ¿qué eje antiguo se convierte en el nuevo eje 0?</div>

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print photo.shape and cells_img.shape.
# 2. Explain why cells_img has no colour axis.
#
# ES:
# 1. Imprime photo.shape y cells_img.shape.
# 2. Explica por qué cells_img no tiene eje de color.
#
# TODO 2 / TAREA 2
#
# EN:
# Convert photo from (H, W, C) to (C, H, W) with np.transpose.
# Name every output axis.
#
# ES:
# Convierte photo de (H, W, C) a (C, H, W) con np.transpose.
# Nombra cada eje de salida.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("photo / foto:", photo.shape)
print("cells_img / microscopía:", cells_img.shape)
print()

print("EN: photo has three axes: height, width, colour.")
print("ES: photo tiene tres ejes: alto, ancho y color.")
print("EN: cells_img is grayscale, so it stores only height and width.")
print("ES: cells_img es en escala de grises, por eso solo almacena alto y ancho.")
print()

chw = np.transpose(photo, (2, 0, 1))

print("HWC:", photo.shape)
print("CHW:", chw.shape)
print()
print("EN: new axis 0 <- old axis 2 (colour)")
print("ES: nuevo eje 0 <- eje antiguo 2 (color)")
print("EN: new axis 1 <- old axis 0 (height)")
print("ES: nuevo eje 1 <- eje antiguo 0 (alto)")
print("EN: new axis 2 <- old axis 1 (width)")
print("ES: nuevo eje 2 <- eje antiguo 1 (ancho)")

assert np.array_equal(chw, np.moveaxis(photo, 2, 0))

### HWC ↔ CHW explorer / Explorador HWC ↔ CHW

Pick a colour channel. The left panel reads it from the original `HWC`. The
right panel reads **the same measured channel** from the transposed `CHW`.

If the transpose was right, the two are identical.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un canal: el panel izquierdo lo lee desde <code>HWC</code>, el derecho lee <b>el mismo canal medido</b> desde <code>CHW</code>. Si la transposición es correcta, ambos coinciden exactamente.</div>

In [ ]:
#@title 🖼️ HWC ↔ CHW explorer / Explorador HWC ↔ CHW — run me / ejecútame { display-mode: 'form' }

# The HWC->CHW transpose (Exercise 1, TODO 2), recomputed here so this
# explorer runs whether or not the folded solution was executed.
chw = np.transpose(photo, (2, 0, 1))

channel_selector = widgets.ToggleButtons(
    options=[
        ("R · Red / Rojo", 0),
        ("G · Green / Verde", 1),
        ("B · Blue / Azul", 2),
    ],
    value=0,
    description="Channel / Canal:",
    style={"description_width": "120px"},
)

def compare_hwc_chw(channel):
    channel_names = {
        0: "Red / Rojo",
        1: "Green / Verde",
        2: "Blue / Azul",
    }

    from_hwc = photo[:, :, channel]
    from_chw = chw[channel]

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))

    axes[0].imshow(from_hwc, cmap="gray")
    axes[0].set_title(
        f"HWC → photo[:, :, {channel}]\n{channel_names[channel]}"
    )

    axes[1].imshow(from_chw, cmap="gray")
    axes[1].set_title(
        f"CHW → chw[{channel}]\n{channel_names[channel]}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured values / Mismos valores medidos:",
        np.array_equal(from_hwc, from_chw),
    )
    print("EN: transpose moved the colour axis; it did not change the pixel values.")
    print("ES: transpose movió el eje de color; no cambió los valores de los píxeles.")

channel_output = widgets.interactive_output(
    compare_hwc_chw,
    {"channel": channel_selector},
)

display(widgets.VBox([channel_selector, channel_output]))

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

`np.transpose(photo, (2, 0, 1))` does not mean “make a shape `(3, 512, 512)`”.

It means: **put old axis 2 first, then old axis 0, then old axis 1**.

The original order was `(H, W, C)`, so the new one is `(C, H, W)`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>np.transpose(photo, (2, 0, 1))</code> no significa «crea una forma <code>(3, 512, 512)</code>». Significa: <b>coloca primero el eje antiguo 2, después el 0 y por último el 1</b>. El orden original era <code>(H, W, C)</code>, así que el nuevo es <code>(C, H, W)</code>.</div>

</details>

## 4.2 Add a batch axis / Agrega un eje de lote

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

One RGB image is `(H, W, C)`. A batch of them adds an axis: `(N, H, W, C)`,
where `N` says **which image**.

We build a real batch from histology, the astronaut and the coffee. Their
original sizes differ, so each contributes a deterministic `256 × 256` centre
crop.

No pixel value is invented.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una imagen RGB es <code>(H, W, C)</code>; un lote añade un eje, <code>(N, H, W, C)</code>, donde <code>N</code> dice <b>qué imagen</b>. Como los tamaños originales difieren, tomamos un recorte central real de <code>256 × 256</code> de cada una. No se inventa ningún píxel.</div>

## Exercise 2 — a real batch, two axes of size 3 / Ejercicio 2 — un lote real, dos ejes de tamaño 3

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Stacked: <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">NHWC = (3, 256, 256, 3)</span>. Transposed:
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">NCHW = (3, 3, 256, 256)</span>.

Two axes now have size 3.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">SAME NUMBER, DIFFERENT QUESTION · MISMO NÚMERO, OTRA PREGUNTA</div><div style="margin:.55em 0"><b>axis 0</b> — which of the three <b>images</b>?</div><div style="margin:.55em 0"><b>axis 1</b> — which of the three <b>colour channels</b>?</div></div>

The number `3` carries none of that.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos ejes tienen tamaño <code>3</code>: uno responde «¿cuál de las tres imágenes?» y el otro «¿cuál de los tres canales?». El número por sí solo no lo dice.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Centre-crop every image in rgb_sources to 256×256.
# 2. Stack them into one batch.
# 3. Verify shape (3, 256, 256, 3).
# 4. Explain the meaning of N, H, W, and C.
#
# ES:
# 1. Recorta al centro cada imagen de rgb_sources a 256×256.
# 2. Apílalas en un solo lote.
# 3. Verifica la forma (3, 256, 256, 3).
# 4. Explica el significado de N, H, W y C.
#
# TODO 4 / TAREA 4
#
# EN:
# Convert NHWC to NCHW.
# Explain why axis 0 and axis 1 both have size 3 but different meanings.
#
# ES:
# Convierte NHWC a NCHW.
# Explica por qué los ejes 0 y 1 tienen tamaño 3 pero significados diferentes.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

batch = np.stack([center_crop_rgb(img) for img in rgb_sources])

print("NHWC:", batch.shape)
print("EN: N=3 images, H=256, W=256, C=3 colour channels.")
print("ES: N=3 imágenes, H=256, W=256, C=3 canales de color.")
print()

nchw = np.transpose(batch, (0, 3, 1, 2))

print("NCHW:", nchw.shape)
print("EN: axis 0 = image; axis 1 = colour.")
print("ES: eje 0 = imagen; eje 1 = color.")
print()

one_image = nchw[0]
red_all_images = nchw[:, 0]

print("nchw[0].shape:", one_image.shape)
print("EN: one image, all three channels.")
print("ES: una imagen, sus tres canales.")
print()

print("nchw[:, 0].shape:", red_all_images.shape)
print("EN: red channel from all three images.")
print("ES: canal rojo de las tres imágenes.")

assert batch.shape == (3, 256, 256, 3)
assert nchw.shape == (3, 3, 256, 256)
assert np.array_equal(
    one_image,
    np.transpose(batch[0], (2, 0, 1)),
)
assert np.array_equal(
    red_all_images,
    batch[:, :, :, 0],
)

### Batch-axis explorer / Explorador de ejes del lote

Pick an image `N` and a channel `C`. The notebook reads the same data from
`NHWC` and from `NCHW`.

Both axes have size `3`. The controls keep their roles apart.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una imagen <code>N</code> y un canal <code>C</code>: el cuaderno lee los mismos datos desde <code>NHWC</code> y desde <code>NCHW</code>. Ambos ejes miden <code>3</code>, y los controles mantienen separados sus papeles.</div>

In [ ]:
#@title 🖼️ Batch-axis explorer / Explorador de ejes del lote — run me / ejecútame { display-mode: 'form' }

# The real batch and its NHWC->NCHW transpose (Exercise 2), recomputed here
# so this explorer runs whether or not the folded solution was executed.
batch = np.stack([center_crop_rgb(img) for img in rgb_sources])
nchw = np.transpose(batch, (0, 3, 1, 2))

image_selector = widgets.ToggleButtons(
    options=[
        ("N=0 · Histology / Histología", 0),
        ("N=1 · Astronaut / Astronauta", 1),
        ("N=2 · Coffee / Café", 2),
    ],
    value=0,
    description="Image N / Imagen N:",
    style={"description_width": "130px"},
)

batch_channel_selector = widgets.ToggleButtons(
    options=[
        ("C=0 · Red / Rojo", 0),
        ("C=1 · Green / Verde", 1),
        ("C=2 · Blue / Azul", 2),
    ],
    value=0,
    description="Channel C / Canal C:",
    style={"description_width": "130px"},
)

def explore_batch_axes(image_idx, channel_idx):
    from_nhwc = batch[image_idx, :, :, channel_idx]
    from_nchw = nchw[image_idx, channel_idx, :, :]

    plt.close("all")
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))

    axes[0].imshow(batch[image_idx])
    axes[0].set_title(
        f"N={image_idx}\n{rgb_names[image_idx]}"
    )

    axes[1].imshow(from_nhwc, cmap="gray")
    axes[1].set_title(
        f"NHWC[{image_idx}, :, :, {channel_idx}]"
    )

    axes[2].imshow(from_nchw, cmap="gray")
    axes[2].set_title(
        f"NCHW[{image_idx}, {channel_idx}, :, :]"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured plane / Mismo plano medido:",
        np.array_equal(from_nhwc, from_nchw),
    )
    print(f"N={image_idx} -> EN: which image | ES: qué imagen")
    print(f"C={channel_idx} -> EN: which colour | ES: qué color")

batch_output = widgets.interactive_output(
    explore_batch_axes,
    {
        "image_idx": image_selector,
        "channel_idx": batch_channel_selector,
    },
)

display(
    widgets.VBox([
        image_selector,
        batch_channel_selector,
        batch_output,
    ])
)

<details>
<summary><strong>Why two size-3 axes are different / Por qué dos ejes de tamaño 3 son diferentes</strong></summary>

In `(N, C, H, W) = (3, 3, 256, 256)` the first `3` answers *which of the three
images*, and the second answers *which of the three RGB channels*.

A shape records sizes. It does not record labels.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>En <code>(N, C, H, W) = (3, 3, 256, 256)</code>, el primer <code>3</code> responde «¿cuál de las tres imágenes?» y el segundo «¿cuál de los tres canales RGB?». La forma guarda tamaños, no etiquetas.</div>

</details>

## 4.3 Reshape vs. transpose / Reshape vs. transpose

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

The most important part of this notebook.

From `photo.shape = (512, 512, 3)`, both of these produce `(3, 512, 512)`:

- `np.transpose(photo, (2, 0, 1))` — correct;
- `photo.reshape(3, 512, 512)` — same shape, scrambled image.

Read the elements out in order and the difference is exact.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE WHOLE DISTINCTION · TODA LA DIFERENCIA</div><div style="margin:.55em 0"><b>reshape</b> changes the shape and nothing else. The flat sequence is untouched: <code>a.reshape(...).ravel()</code> always equals <code>a.ravel()</code>. It moves the brackets.</div><div style="margin:.55em 0"><b>transpose</b> changes the shape <i>and</i> reorders that flat sequence. <code>a.transpose(...).ravel()</code> is a permutation of <code>a.ravel()</code> — same values, different reading order.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><b>reshape</b> cambia la forma y nada más: <code>a.reshape(...).ravel()</code> siempre es igual a <code>a.ravel()</code>.</li><li style="margin:.35em 0"><b>transpose</b> cambia la forma <i>y</i> reordena esa secuencia plana: <code>a.transpose(...).ravel()</code> es una permutación de <code>a.ravel()</code>.</li></ul></div>

### One implementation note / Una nota de implementación

NumPy does not move memory to transpose. It hands back a view with permuted
**strides**. The buffer is untouched, and the reordering is paid for later — by
the first operation that forces a copy: `ravel`, `copy`, `ascontiguousarray`,
or a library demanding contiguous input.

Free to write. Not always free to run.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE LIBRARY · LA BIBLIOTECA</div>Books arranged as <b>shelf × position × category</b>. <code>transpose</code> moves the labelled dimensions. <code>reshape</code> takes the books in their current reading order and fills a differently shaped unit, labels be damned.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>NumPy no mueve memoria al transponer: devuelve una vista con <b>strides</b> permutados. El reordenamiento se paga después, en la primera operación que obliga a copiar — <code>ravel</code>, <code>copy</code>, <code>ascontiguousarray</code>, o una librería que exija datos contiguos.<br><br>Libros ordenados por <b>estante × posición × categoría</b>: <code>transpose</code> mueve las dimensiones etiquetadas; <code>reshape</code> toma los libros en su orden actual y llena una estantería de otra forma, ignorando las etiquetas.</div>

In [ ]:
# The claim above, in six values. Deliberately synthetic and tiny: the point is
# the reading order, and 0..5 is the only data you can check by eye.
a = np.arange(6).reshape(2, 3)
print("a =\n", a, "\n")

print("a.ravel()               ", a.ravel())              # 0 1 2 3 4 5
print("a.reshape(3, 2).ravel() ", a.reshape(3, 2).ravel())  # unchanged
print("a.T.ravel()             ", a.T.ravel())            # reordered
print()
print("shapes  a.reshape(3, 2):", a.reshape(3, 2).shape, " a.T:", a.T.shape)
print("strides a:", a.strides, " a.T:", a.T.strides,
      "— transpose permuted the strides, not the buffer")

## Exercise 3 — code that runs and is still wrong / Ejercicio 3 — código que funciona y aun así está mal

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>Two arrays share the shape <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(3, 512, 512)</span>.<div style='margin-top:10px'>Does that guarantee <code>array_a[0]</code> and <code>array_b[0]</code> are the same colour channel?</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos arreglos tienen la misma forma <code>(3, 512, 512)</code>. ¿Eso garantiza que <code>array_a[0]</code> y <code>array_b[0]</code> sean el mismo canal de color?</div>

In [ ]:
# TODO 5 / TAREA 5
#
# EN:
# 1. Compute the correct CHW array with transpose.
# 2. Compute photo.reshape(3, 512, 512).
# 3. Verify that the shapes are equal.
# 4. Verify whether the arrays themselves are equal.
# 5. Display plane 0 from both arrays.
# 6. Explain why reshape cannot replace transpose here.
#
# ES:
# 1. Calcula el arreglo CHW correcto con transpose.
# 2. Calcula photo.reshape(3, 512, 512).
# 3. Verifica que las formas sean iguales.
# 4. Verifica si los arreglos completos son iguales.
# 5. Muestra el plano 0 de ambos.
# 6. Explica por qué reshape no puede sustituir transpose en este caso.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

print("Correct CHW / CHW correcto:", correct_chw.shape)
print("Reshape output / Salida reshape:", wrong_reshape.shape)
print()

print(
    "Same shape / Misma forma:",
    correct_chw.shape == wrong_reshape.shape,
)
print(
    "Same data arrangement / Misma organización de datos:",
    np.array_equal(correct_chw, wrong_reshape),
)

difference_fraction = np.mean(correct_chw != wrong_reshape)

print(
    f"Different positions / Posiciones diferentes: "
    f"{difference_fraction:.1%}"
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

axes[0].imshow(photo)
axes[0].set_title("Original RGB / RGB original")

axes[1].imshow(correct_chw[0], cmap="gray")
axes[1].set_title(
    "transpose\nreal red channel / canal rojo real"
)

axes[2].imshow(wrong_reshape[0], cmap="gray")
axes[2].set_title(
    "reshape\nscrambled interpretation / interpretación alterada"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print()
print("EN: transpose preserved the meaning of the colour axis.")
print("ES: transpose conservó el significado del eje de color.")
print("EN: reshape reached the same shape but did not move the colour axis correctly.")
print("ES: reshape alcanzó la misma forma, pero no movió correctamente el eje de color.")

### Transpose-vs-reshape explorer / Explorador transpose contra reshape

Pick a plane — `0`, `1` or `2` — and the method that produced
`(3, 512, 512)`. Compare against the true RGB channel.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un plano — <code>0</code>, <code>1</code> o <code>2</code> — y el método que produjo <code>(3, 512, 512)</code>. Compara con el canal RGB verdadero.</div>

In [ ]:
#@title 🔬 Transpose vs reshape / Transpose contra reshape — run me / ejecútame { display-mode: 'form' }

# The correct transpose and the same-shape reshape (Exercise 3), recomputed
# here so this explorer runs whether or not the folded solution was executed.
correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

plane_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=2,
    step=1,
    description="Plane / Plano:",
    continuous_update=False,
    style={"description_width": "100px"},
)

method_toggle = widgets.ToggleButtons(
    options=[
        ("Transpose / Transposición", "transpose"),
        ("Reshape", "reshape"),
    ],
    value="transpose",
    description="Method / Método:",
    style={"description_width": "110px"},
)

def explore_method(plane, method):
    true_channel = photo[:, :, plane]

    if method == "transpose":
        candidate = correct_chw[plane]
        method_name = "transpose / transposición"
    else:
        candidate = wrong_reshape[plane]
        method_name = "reshape"

    same = np.array_equal(candidate, true_channel)
    mae = float(
        np.mean(
            np.abs(
                candidate.astype(np.float32)
                - true_channel.astype(np.float32)
            )
        )
    )

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5))

    axes[0].imshow(true_channel, cmap="gray")
    axes[0].set_title(
        f"True channel {plane} / Canal real {plane}"
    )

    axes[1].imshow(candidate, cmap="gray")
    axes[1].set_title(
        f"{method_name}\nplane/plano {plane}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Exact match / Coincidencia exacta:", same)
    print(f"Mean absolute difference / Diferencia absoluta media: {mae:.3f}")

    if same:
        print("EN: this operation preserved the intended channel semantics.")
        print("ES: esta operación conservó el significado correcto del canal.")
    else:
        print("EN: the shape is plausible, but the channel semantics are wrong.")
        print("ES: la forma parece correcta, pero el significado del canal es incorrecto.")

method_output = widgets.interactive_output(
    explore_method,
    {
        "plane": plane_slider,
        "method": method_toggle,
    },
)

display(
    widgets.VBox([
        widgets.HBox([plane_slider, method_toggle]),
        method_output,
    ])
)

## Which one do I want? / ¿Cuál necesito?

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE RULE · LA REGLA</div><div style="margin:.55em 0"><b>Move axes → transpose.</b> <code>HWC → CHW</code>, <code>NHWC → NCHW</code>: the semantic positions change.</div><div style="margin:.55em 0"><b>Group dimensions → reshape.</b> <code>(H, W) → (H×W,)</code>: you regroup on purpose, and no axis pretends to have moved.</div></div>

And a test you can actually run: **`reshape` never changes `ravel()`;
`transpose` always does.**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Mover ejes → transpose. Agrupar dimensiones → reshape.</b> O como comprobación ejecutable: <code>reshape</code> nunca cambia <code>ravel()</code>, <code>transpose</code> siempre lo hace.</div>

## Quick reasoning challenge / Reto rápido de razonamiento

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

**Transpose** or **reshape**? Decide before you run.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">FOUR TASKS · CUATRO TAREAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><code>(512,512,3) → (3,512,512)</code>, because a model wants colour first.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><code>(8,8) → (64,)</code>, because one digit should become one feature vector.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><code>(3,256,256,3) → (3,3,256,256)</code>, because the framework wants channel before height.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><code>(1797,8,8) → (1797,64)</code>, because each image should become one row.</div></div>

The question is never only “what shape do I want?” It is **“do I want to move
axes, or group dimensions?”**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pregunta nunca es solo «¿qué forma quiero?», sino <b>«¿quiero mover ejes o agrupar dimensiones?»</b></div>

In [ ]:
#@title 🧠 Reasoning challenge / Reto de razonamiento — run me / ejecútame { display-mode: 'form' }

reasoning_task = widgets.Dropdown(
    options=[
        ("1 · HWC → CHW", 1),
        ("2 · 8×8 image → 64 values / imagen 8×8 → 64 valores", 2),
        ("3 · NHWC → NCHW", 3),
        ("4 · (1797,8,8) → (1797,64)", 4),
    ],
    value=1,
    description="Task / Tarea:",
    style={"description_width": "100px"},
)

def explain_choice(task):
    answers = {
        1: (
            "transpose",
            "colour must move from the last axis to the first",
            "el color debe moverse del último eje al primero",
        ),
        2: (
            "reshape",
            "height and width are intentionally grouped into one feature axis",
            "alto y ancho se agrupan intencionalmente en un eje de características",
        ),
        3: (
            "transpose",
            "the colour axis must move before the spatial axes",
            "el eje de color debe moverse antes de los ejes espaciales",
        ),
        4: (
            "reshape",
            "each 8×8 image is intentionally flattened into 64 values",
            "cada imagen 8×8 se aplana intencionalmente en 64 valores",
        ),
    }

    operation, en, es = answers[task]

    print("Operation / Operación:", operation)
    print("EN:", en)
    print("ES:", es)

reasoning_output = widgets.interactive_output(
    explain_choice,
    {"task": reasoning_task},
)

display(widgets.VBox([reasoning_task, reasoning_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Every value you touched was a measured pixel.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">FIVE IDEAS · CINCO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Shape is not semantics.</b> Two axes of size <code>3</code> can mean different things.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>HWC and CHW</b> hold the same image in a different axis order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>NHWC and NCHW</b> hold the same batch in a different axis order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Transpose moves axes.</b></div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span><b>Reshape regroups stored values.</b> It can hit the shape you wanted and the meaning you did not.</div></div>

Move axes when axis meaning changes. Reshape when you regroup on purpose — and
never let a reshape pretend an axis moved.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> la forma no es la semántica; <b>2 ·</b> <code>HWC</code> y <code>CHW</code> son la misma imagen en otro orden; <b>3 ·</b> igual con <code>NHWC</code> y <code>NCHW</code>; <b>4 ·</b> transpose mueve ejes; <b>5 ·</b> reshape reagrupa valores y puede acertar la forma y errar el significado.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

## Time for Kahoot 🎯 / Hora de Kahoot 🎯

**Kahoot 1 — Tensor Vocabulary & Shapes / Vocabulario de tensores y formas**  
6 questions / 6 preguntas · about 5 minutes / unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Entra a <b>kahoot.it</b> con el PIN que aparece en la pantalla del facilitador.</div></div>

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-1)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_1_vocabulary_shapes.xlsx)

Next / Siguiente: **05 · Video pipeline design / Diseño de un pipeline de vídeo** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)